In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python utilizado:", sys.executable)

Python utilizado: c:\Users\CES\Documents\DATA_ENGINEER\CURSO_SPARK\.venv\Scripts\python.exe


# 03 — Inspección de DataFrames en PySpark

## 1. Objetivo

Aprender a inspeccionar la estructura y el contenido de un DataFrame antes de
realizar transformaciones.

En este notebook se estudiarán:

- `show()`
- `printSchema()`
- `columns`
- `dtypes`
- `count()`

Estas herramientas permiten conocer:

- qué datos contiene el DataFrame;
- cuáles son sus columnas;
- qué tipos de datos tiene;
- cuántos registros contiene;
- si la estructura coincide con lo esperado por el negocio.

## 2. Concepto oficial

En PySpark, un DataFrame representa una colección de datos organizada en
columnas con nombres.

Antes de transformar un DataFrame es recomendable inspeccionar:

1. una muestra de sus registros;
2. los nombres de sus columnas;
3. los tipos de datos;
4. su schema;
5. la cantidad de filas.

La inspección permite detectar problemas como:

- columnas con nombres incorrectos;
- tipos de datos inesperados;
- archivos vacíos;
- estructuras diferentes a las definidas por el negocio.

In [2]:
from pyspark.sql import SparkSession

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DecimalType,DateType, BooleanType, IntegerType

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("03_inspeccion_dataframe")
    .master("local[2]")
    .config("spark.python.worker.reuse", "false")
    .getOrCreate()
)

print(spark.version)

4.2.0


In [5]:
schema_customers = StructType([
    StructField("customer_id", StringType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("credit_score", IntegerType(), True),
    StructField("created_at", DateType(), True),
])

In [6]:
ruta_customers = "../data/raw/customers.csv"
df_customers = spark.read.csv(
    ruta_customers,
    header=True,
    schema=schema_customers
)

df_customers.show(5, truncate=False)

+---------------+----------+---------+--------------------------+------------------+------------+----------+
|customer_id    |first_name|last_name|email                     |city              |credit_score|created_at|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
|CUSXAJI0Y6DPBHS|Kevin     |Young    |brauncameron@example.net  |North Williamville|327         |2025-04-17|
|CUSHXTHV3A3ZMF8|Jason     |Clements |toddwilliam@example.net   |Martinezside      |644         |2020-02-23|
|CUSDD4V30T9NT3W|Randy     |Thompson |trevoranderson@example.org|Gallowayfurt      |670         |2025-06-22|
|CUSGCX1945NQ4FM|Laura     |Phillips |valdezgeorge@example.com  |Morrisview        |573         |2019-10-20|
|CUSVG0FN9XUY41I|Savannah  |Swanson  |smithluis@example.com     |Lake Anna         |332         |2022-07-16|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
only showing top 5 

In [7]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: date (nullable = true)



## 4. Método `show()`

El método `show()` imprime una muestra de las filas del DataFrame en la salida
del notebook.

Su sintaxis básica es:

`df.show()`

Por defecto, Spark muestra hasta 20 filas y puede recortar textos largos.

También podemos indicar:

- la cantidad de filas;
- si queremos evitar el truncamiento.

Ejemplo:

`df.show(5, truncate=False)`

In [8]:
df_customers.show(5, truncate=False)

+---------------+----------+---------+--------------------------+------------------+------------+----------+
|customer_id    |first_name|last_name|email                     |city              |credit_score|created_at|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
|CUSXAJI0Y6DPBHS|Kevin     |Young    |brauncameron@example.net  |North Williamville|327         |2025-04-17|
|CUSHXTHV3A3ZMF8|Jason     |Clements |toddwilliam@example.net   |Martinezside      |644         |2020-02-23|
|CUSDD4V30T9NT3W|Randy     |Thompson |trevoranderson@example.org|Gallowayfurt      |670         |2025-06-22|
|CUSGCX1945NQ4FM|Laura     |Phillips |valdezgeorge@example.com  |Morrisview        |573         |2019-10-20|
|CUSVG0FN9XUY41I|Savannah  |Swanson  |smithluis@example.com     |Lake Anna         |332         |2022-07-16|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
only showing top 5 

### Resultado del ejemplo con `show()`

El método `show(5, truncate=False)` mostró los primeros cinco registros del
DataFrame.

Se pudieron observar:

- los nombres de las columnas;
- una muestra de los datos;
- la distribución visual de las filas;
- los textos completos sin truncamiento.

`show()` permite revisar rápidamente el contenido del DataFrame, pero no muestra
formalmente los tipos de datos de cada columna.

In [9]:
df_customers.show()

+---------------+----------+---------+--------------------+--------------------+------------+----------+
|    customer_id|first_name|last_name|               email|                city|credit_score|created_at|
+---------------+----------+---------+--------------------+--------------------+------------+----------+
|CUSXAJI0Y6DPBHS|     Kevin|    Young|brauncameron@exam...|  North Williamville|         327|2025-04-17|
|CUSHXTHV3A3ZMF8|     Jason| Clements|toddwilliam@examp...|        Martinezside|         644|2020-02-23|
|CUSDD4V30T9NT3W|     Randy| Thompson|trevoranderson@ex...|        Gallowayfurt|         670|2025-06-22|
|CUSGCX1945NQ4FM|     Laura| Phillips|valdezgeorge@exam...|          Morrisview|         573|2019-10-20|
|CUSVG0FN9XUY41I|  Savannah|  Swanson|smithluis@example...|           Lake Anna|         332|2022-07-16|
|CUSOC6UZHR5XFF0|    Ashley|     Wang|reesekendra@examp...|             Amybury|         569|2025-07-22|
|CUSPVN9ER14FFYV|    Morgan|   Miller| narnold@example.

In [10]:
df_customers.show(3)

+---------------+----------+---------+--------------------+------------------+------------+----------+
|    customer_id|first_name|last_name|               email|              city|credit_score|created_at|
+---------------+----------+---------+--------------------+------------------+------------+----------+
|CUSXAJI0Y6DPBHS|     Kevin|    Young|brauncameron@exam...|North Williamville|         327|2025-04-17|
|CUSHXTHV3A3ZMF8|     Jason| Clements|toddwilliam@examp...|      Martinezside|         644|2020-02-23|
|CUSDD4V30T9NT3W|     Randy| Thompson|trevoranderson@ex...|      Gallowayfurt|         670|2025-06-22|
+---------------+----------+---------+--------------------+------------------+------------+----------+
only showing top 3 rows


In [11]:
df_customers.show(3, truncate=False)

+---------------+----------+---------+--------------------------+------------------+------------+----------+
|customer_id    |first_name|last_name|email                     |city              |credit_score|created_at|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
|CUSXAJI0Y6DPBHS|Kevin     |Young    |brauncameron@example.net  |North Williamville|327         |2025-04-17|
|CUSHXTHV3A3ZMF8|Jason     |Clements |toddwilliam@example.net   |Martinezside      |644         |2020-02-23|
|CUSDD4V30T9NT3W|Randy     |Thompson |trevoranderson@example.org|Gallowayfurt      |670         |2025-06-22|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
only showing top 3 rows


### Análisis del ejercicio 1

1. ¿Cuántas filas intentó mostrar `show()` sin parámetros?
Rta: 20
2. ¿Cuántas filas mostró `show(3)`?
Rta: 3
3. ¿Qué diferencia observaste al utilizar `truncate=False`?
Rta, que email esta completo
4. ¿`show()` modifica el DataFrame original?
Rta: No

## 5. Método `printSchema()`

El método `printSchema()` muestra la estructura del DataFrame.

Permite observar:

- los nombres de las columnas;
- los tipos de datos;
- si las columnas aparecen como anulables;
- la estructura jerárquica del schema.

Sintaxis:

`df.printSchema()`

In [12]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: date (nullable = true)



### Análisis de `printSchema()`

1. ¿Qué tipo de dato tiene `customer_id`?
Rta: String
2. ¿Qué tipo de dato tiene `credit_score`?
Rta: Integer
3. ¿Qué tipo de dato tiene `created_at`?
Rta: date
4. ¿Qué significa `nullable = true`?
Rta: Que el atributo permite valores nulos

In [13]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: date (nullable = true)



### Ejercicio 2 — Interpretación del schema

1. ¿Cuántas columnas tiene el DataFrame?
Rta: 7 columnas
2. ¿Cuáles columnas son de tipo `string`?
Rta: desde customer_id hasta city
3. ¿Cuál columna es de tipo `integer`?
Rta: credit_score
4. ¿Cuál columna es de tipo `date`?
created_at
5. ¿`printSchema()` muestra los valores de las filas?
No, muestra el nombre de las columnas, el tipo de dato y si permite nulos o no
6. ¿`printSchema()` modifica el DataFrame?
No

## 6. Atributo `columns`

El atributo `columns` devuelve una lista con los nombres de todas las columnas
del DataFrame.

Sintaxis:

`df.columns`

A diferencia de `show()` y `printSchema()`, `columns` no utiliza paréntesis
porque es un atributo y no un método.

In [14]:
df_customers.columns

['customer_id',
 'first_name',
 'last_name',
 'email',
 'city',
 'credit_score',
 'created_at']

### Análisis de `columns`

1. ¿Qué tipo de estructura devolvió: lista, tupla o diccionario?
Rta: una lista con los nombres de los atributos.
2. ¿Cuántos nombres aparecen?
Rta: 7
3. ¿Cuál es la primera columna?
Rta: customer_id
4. ¿Cuál es la última columna?
Rta: created_at
5. ¿Por qué `columns` se escribe sin paréntesis?
Por que es un atributo y no una función.

Ejercicio 3 de 10 — Inspección con columns

In [15]:
columnas_customers = df_customers.columns
print(columnas_customers)
print(len(columnas_customers))
print(columnas_customers[0])
print(columnas_customers[-1])

['customer_id', 'first_name', 'last_name', 'email', 'city', 'credit_score', 'created_at']
7
customer_id
created_at


### Ejercicio 3 — Uso de `columns`

1. ¿Qué información quedó guardada en `columnas_customers`?
Rta: Una lista con los nombres de los atributos.
2. ¿Qué resultado devolvió `len(columnas_customers)`?
7
3. ¿Qué columna devolvió `columnas_customers[0]`?
customer_id
4. ¿Qué columna devolvió `columnas_customers[-1]`?
la ultima created_at
5. ¿Este ejercicio consultó datos de las filas o solamente la estructura?
solo estructura

## 7. Atributo `dtypes`

El atributo `dtypes` devuelve una lista con la información de las columnas y
sus tipos de datos.

Cada elemento contiene:

- el nombre de la columna;
- el tipo de dato representado como texto.

Sintaxis:

`df.dtypes`

Al igual que `columns`, se escribe sin paréntesis porque es un atributo.

In [16]:
df_customers.dtypes

[('customer_id', 'string'),
 ('first_name', 'string'),
 ('last_name', 'string'),
 ('email', 'string'),
 ('city', 'string'),
 ('credit_score', 'int'),
 ('created_at', 'date')]

### Análisis de `dtypes`

1. ¿Qué tipo de estructura devolvió `dtypes`?
Rta: Una lista con 7 tuplas y en cada tupla se identifican el nombre de la columna y el tipo de dato de la columna.
2. ¿Qué información contiene cada elemento?
Rta: en cada tupla se identifican el nombre de la columna y el tipo de dato de la columna
3. ¿Qué tipo aparece para `credit_score`?
Rta: int
4. ¿Qué tipo aparece para `created_at`?
Rta: date
5. ¿Qué diferencia observas entre `columns` y `dtypes`?
Rta: columns retorna una lista con los nombres de las columnas, y dtypes retorna una lista de tuplas las cuales contienen el nombre de la columna y el tipo de dato de la misma.

In [17]:
tipos_customers = df_customers.dtypes
print(tipos_customers)
print(len(tipos_customers))
print(tipos_customers[0])
print(tipos_customers[-1])
print(tipos_customers[0][0])#trae el nombre de la columna
print(tipos_customers[-1][1])#trae el tipo de dato de la columna

[('customer_id', 'string'), ('first_name', 'string'), ('last_name', 'string'), ('email', 'string'), ('city', 'string'), ('credit_score', 'int'), ('created_at', 'date')]
7
('customer_id', 'string')
('created_at', 'date')
customer_id
date


### Ejercicio 4 — Uso de `dtypes`

1. En `tipos_customers` quedó guardada una lista de tuplas con los nombres de
   las columnas y sus tipos de datos.

2. La lista contiene 7 elementos.

3. `tipos_customers[0]` contiene la primera tupla:
   `('customer_id', 'string')`.

4. `tipos_customers[-1]` contiene la última tupla:
   `('created_at', 'date')`.

5. Cada elemento de la lista representa una columna.

6. `tipos_customers[0][0]` devolvió el primer valor de la primera tupla:
   `customer_id`.

7. `tipos_customers[-1][1]` devolvió el segundo valor de la última tupla:
   `date`.

## 8. Método `count()`

El método `count()` devuelve la cantidad total de filas del DataFrame.

Sintaxis:

`df.count()`

A diferencia de `columns` y `dtypes`, `count()` sí utiliza paréntesis porque es
un método.

`count()` no muestra una muestra de datos: realiza el conteo total de registros.

In [18]:
cantidad_customers = df_customers.count()
print(cantidad_customers)

50000


### Análisis de `count()`

1. ¿Cuántas filas tiene `df_customers`?
Rta: 50000
2. ¿El resultado de `count()` es una lista o un número?
Rta: un numero
3. ¿`count()` cuenta columnas o filas?
Rta: filas
4. ¿`count()` modifica el DataFrame?
Rta: No
5. ¿Qué diferencia existe entre `show()` y `count()`?
Show muestra los datos y count muestra la cantidad de registros del df

### Análisis de `count()`

1. `df_customers` tiene 50.000 filas.
2. `count()` devuelve un número.
3. `count()` cuenta filas.
4. `count()` no modifica el DataFrame.
5. `show()` muestra una muestra de los datos, mientras que `count()` devuelve
   la cantidad total de registros.

In [19]:
print("Columnas:", df_customers.columns)
print("Tipos:", df_customers.dtypes)
print("Cantidad de filas:", df_customers.count())

df_customers.show(2, truncate=False)
df_customers.printSchema()

Columnas: ['customer_id', 'first_name', 'last_name', 'email', 'city', 'credit_score', 'created_at']
Tipos: [('customer_id', 'string'), ('first_name', 'string'), ('last_name', 'string'), ('email', 'string'), ('city', 'string'), ('credit_score', 'int'), ('created_at', 'date')]
Cantidad de filas: 50000
+---------------+----------+---------+------------------------+------------------+------------+----------+
|customer_id    |first_name|last_name|email                   |city              |credit_score|created_at|
+---------------+----------+---------+------------------------+------------------+------------+----------+
|CUSXAJI0Y6DPBHS|Kevin     |Young    |brauncameron@example.net|North Williamville|327         |2025-04-17|
|CUSHXTHV3A3ZMF8|Jason     |Clements |toddwilliam@example.net |Martinezside      |644         |2020-02-23|
+---------------+----------+---------+------------------------+------------------+------------+----------+
only showing top 2 rows
root
 |-- customer_id: string (nu

### Ejercicio 5 — Comparación de herramientas de inspección

1. ¿Qué herramienta utilizaste para ver los nombres de las columnas?  
   `df_customers.columns`

2. ¿Qué herramienta utilizaste para ver nombres y tipos?  
   `df_customers.dtypes`

3. ¿Qué herramienta utilizaste para conocer la cantidad de filas?  
   `df_customers.count()`

4. ¿Qué herramienta utilizaste para ver una muestra de datos?  
   `df_customers.show()`

5. ¿Qué herramienta utilizaste para ver el schema completo?  
   `df_customers.printSchema()`

6. ¿Cuál de estas herramientas modificaría el DataFrame?  
   Ninguna. Todas sirven para inspeccionar o consultar información del
   DataFrame sin modificarlo.

In [20]:
cantidad_filas = df_customers.count()
cantidad_columnas = len(df_customers.columns)

print("Cantidad de filas:", cantidad_filas)
print("Cantidad de columnas:", cantidad_columnas)
print("Primera columna:", df_customers.columns[0])
print("Última columna:", df_customers.columns[-1])
print("Primer tipo:", df_customers.dtypes[0])
print("Último tipo:", df_customers.dtypes[-1])

Cantidad de filas: 50000
Cantidad de columnas: 7
Primera columna: customer_id
Última columna: created_at
Primer tipo: ('customer_id', 'string')
Último tipo: ('created_at', 'date')


### Ejercicio 6 — Reporte básico de inspección

1. ¿Cuántas filas tiene el DataFrame?  
   50.000.

2. ¿Cuántas columnas tiene?  
   7.

3. ¿Cuál es la primera columna?  
   `customer_id`.

4. ¿Cuál es la última columna?  
   `created_at`.

5. ¿Qué representa el resultado de `df_customers.dtypes[0]`?  
   Representa la primera tupla de la lista devuelta por `dtypes`. Esa tupla
   contiene el nombre de la primera columna y su tipo de dato.

6. ¿Qué diferencia existe entre `count()` y `len(df_customers.columns)`?  
   `count()` devuelve la cantidad de filas del DataFrame, mientras que
   `len(df_customers.columns)` devuelve la cantidad de columnas.

7. ¿Cuál de las dos instrucciones cuenta registros?  
   `count()`.

8. ¿Cuál de las dos instrucciones cuenta columnas?  
   `len(df_customers.columns)`.

In [21]:
data_account_types = [
    {
        "account_type": "AHORROS",
        "description": "Cuenta de ahorro",
        "allows_overdraft": False,
    },
    {
        "account_type": "CORRIENTE",
        "description": "Cuenta de corriente",
        "allows_overdraft": True,
    },
    {
        "account_type": "INVERSION",
        "description": "Cuenta de inversión",
        "allows_overdraft": True,
    },
]

df_account_types = spark.createDataFrame(data_account_types)

df_account_types.show(truncate=False)
df_account_types.printSchema()

print("Columnas:", df_account_types.columns)
print("Tipos:", df_account_types.dtypes)
print("Cantidad de filas:", df_account_types.count())
print("Cantidad de columnas:", len(df_account_types.columns))

+------------+----------------+-------------------+
|account_type|allows_overdraft|description        |
+------------+----------------+-------------------+
|AHORROS     |false           |Cuenta de ahorro   |
|CORRIENTE   |true            |Cuenta de corriente|
|INVERSION   |true            |Cuenta de inversión|
+------------+----------------+-------------------+

root
 |-- account_type: string (nullable = true)
 |-- allows_overdraft: boolean (nullable = true)
 |-- description: string (nullable = true)

Columnas: ['account_type', 'allows_overdraft', 'description']
Tipos: [('account_type', 'string'), ('allows_overdraft', 'boolean'), ('description', 'string')]
Cantidad de filas: 3
Cantidad de columnas: 3


### Ejercicio 7 — Inspección de `df_account_types`

1. ¿Cuántas filas tiene el DataFrame?

3
2. ¿Cuántas columnas tiene?

3
3. ¿Cuáles son los nombres de las columnas?

'account_type', 'allows_overdraft', 'description'
4. ¿Qué tipo de dato tiene `allows_overdraft`?

boolean
5. ¿Cuál herramienta mostró los registros del DataFrame?

.show()
6. ¿Cuál herramienta mostró la estructura completa?

.printSchema()
7. ¿Qué resultado devolvió `df_account_types.dtypes`?

una lista de tuplas:  [('account_type', 'string'), ('allows_overdraft', 'boolean'), ('description', 'string')]
8. ¿Qué resultado devolvió `df_account_types.columns`?

una lista con los nombres de las columnas. ['account_type', 'allows_overdraft', 'description']
9. ¿Las herramientas de inspección dependen del archivo `customers.csv`?

No, dependen del df creado
10. ¿Qué demuestra este ejercicio sobre `show()`, `printSchema()`, `columns`,
    `dtypes` y `count()`?
show() entrega una muestra de los registros del df si no le das parametros te trae 20 y si los valores string son muy largos los trunca.
printSchema() entrega una breve descripcion de los campos del df, nombre de columnas, tipos de datos y nulidad.
columns: retorna una lista con los nombres de las columnas del df
dtypes retorna una lista de tuplas y cada tupla contiene dos objetos uno es el nombre de la columna y el otro es el tipo de la columna.
count() devuelve el conteo de registros total del df

In [25]:
print("CUSTOMERS")
print("Filas:", df_customers.count())
print("Columnas:", len(df_customers.columns))
print("Nombres:", df_customers.columns)
print("Tipos:", df_customers.dtypes)

print()

print("ACCOUNT TYPES")
print("Filas:", df_account_types.count())
print("Columnas:", len(df_account_types.columns))
print("Nombres:", df_account_types.columns)
print("Tipos:", df_account_types.dtypes)

print("Diferencia de filas ", df_customers.count()-df_account_types.count())
print("Diferencia de columnas ", len(df_customers.columns)-len(df_account_types.columns))

CUSTOMERS
Filas: 50000
Columnas: 7
Nombres: ['customer_id', 'first_name', 'last_name', 'email', 'city', 'credit_score', 'created_at']
Tipos: [('customer_id', 'string'), ('first_name', 'string'), ('last_name', 'string'), ('email', 'string'), ('city', 'string'), ('credit_score', 'int'), ('created_at', 'date')]

ACCOUNT TYPES
Filas: 3
Columnas: 3
Nombres: ['account_type', 'allows_overdraft', 'description']
Tipos: [('account_type', 'string'), ('allows_overdraft', 'boolean'), ('description', 'string')]
Diferencia de filas  49997
Diferencia de columnas  4


### Ejercicio 8 — Comparación entre DataFrames

1. `df_customers` tiene más filas.

2. `df_customers` tiene más columnas.

3. La diferencia es de 49.997 filas.

4. La diferencia es de 4 columnas.

5. La columna de tipo `boolean` es `allows_overdraft`.

6. La columna de tipo `date` es `created_at`.

7. Para contar filas se utilizó `count()`.

8. Para contar columnas se utilizó:

   `len(df.columns)`

   `columns` no lleva paréntesis porque es un atributo.

9. Para comparar los tipos se utilizó `dtypes`.

10. Es importante inspeccionar los DataFrames antes de relacionarlos porque
    pueden no tener columnas compatibles. También se debe validar que las
    columnas utilizadas para relacionarlos tengan tipos de datos compatibles.

In [26]:
print("REPORTE DE INSPECCIÓN")

print("DataFrame:", "df_customers")
print("Filas:", df_customers.count())
print("Columnas:", len(df_customers.columns))
print("Primera columna:", df_customers.columns[0])
print("Última columna:", df_customers.columns[-1])
print("Primer tipo:", df_customers.dtypes[0])
print("Último tipo:", df_customers.dtypes[-1])

print()

print("DataFrame:", "df_account_types")
print("Filas:", df_account_types.count())
print("Columnas:", len(df_account_types.columns))
print("Primera columna:", df_account_types.columns[0])
print("Última columna:", df_account_types.columns[-1])
print("Primer tipo:", df_account_types.dtypes[0])
print("Último tipo:", df_account_types.dtypes[-1])

REPORTE DE INSPECCIÓN
DataFrame: df_customers
Filas: 50000
Columnas: 7
Primera columna: customer_id
Última columna: created_at
Primer tipo: ('customer_id', 'string')
Último tipo: ('created_at', 'date')

DataFrame: df_account_types
Filas: 3
Columnas: 3
Primera columna: account_type
Última columna: description
Primer tipo: ('account_type', 'string')
Último tipo: ('description', 'string')


### Ejercicio 9 — Reporte resumido

1. El reporte permite comparar el nombre del DataFrame, la cantidad de filas,
   la cantidad de columnas, la primera y última columna, y el primer y último
   elemento de `dtypes`.

2. `df_customers` tiene mayor volumen.

3. `df_account_types` tiene una estructura más pequeña.

4. El primer elemento de `dtypes` representa la primera tupla con el nombre de
   la columna y su tipo de dato.

5. El último elemento de `dtypes` representa la última tupla con el nombre de
   la columna y su tipo de dato.

6. El reporte no transforma los datos.

7. `count()` consulta las filas para obtener el total de registros.

8. `columns` y `dtypes` consultan solamente la estructura del DataFrame.

In [27]:
print("INSPECCIÓN INTEGRAL DE df_customers")

df_customers.show(3, truncate=False)
df_customers.printSchema()

print("Columnas:", df_customers.columns)
print("Tipos:", df_customers.dtypes)
print("Cantidad de filas:", df_customers.count())
print("Cantidad de columnas:", len(df_customers.columns))

INSPECCIÓN INTEGRAL DE df_customers
+---------------+----------+---------+--------------------------+------------------+------------+----------+
|customer_id    |first_name|last_name|email                     |city              |credit_score|created_at|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
|CUSXAJI0Y6DPBHS|Kevin     |Young    |brauncameron@example.net  |North Williamville|327         |2025-04-17|
|CUSHXTHV3A3ZMF8|Jason     |Clements |toddwilliam@example.net   |Martinezside      |644         |2020-02-23|
|CUSDD4V30T9NT3W|Randy     |Thompson |trevoranderson@example.org|Gallowayfurt      |670         |2025-06-22|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
only showing top 3 rows
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city

### Ejercicio 10 — Inspección integral

1. `show()` entrega una muestra de los registros del DataFrame.

2. `printSchema()` muestra los nombres de las columnas, sus tipos de datos y si
   aparecen como anulables.

3. `columns` devuelve una lista con los nombres de las columnas.

4. `dtypes` devuelve una lista de tuplas. Cada tupla contiene el nombre de una
   columna y su tipo de dato.

5. `count()` devuelve la cantidad total de filas del DataFrame.

6. Para revisar valores reales utilizaría `df.show()`.

7. Para revisar la nulabilidad utilizaría `df.printSchema()`.

8. Para conocer la cantidad de columnas utilizaría `len(df.columns)`.

9. Para conocer la cantidad de filas utilizaría `df.count()`.

10. Conviene inspeccionar un DataFrame antes de transformarlo para validar su
    estructura, sus tipos, su volumen y una muestra de sus datos. Esto permite
    detectar problemas antes de aplicar reglas de calidad y de negocio.

## Resumen del notebook

1. `show()` permite visualizar una muestra de los registros.
2. `show()` no modifica el DataFrame.
3. `printSchema()` muestra la estructura completa.
4. `columns` devuelve los nombres de las columnas.
5. `dtypes` devuelve nombres y tipos de datos.
6. `count()` devuelve la cantidad total de filas.
7. `len(df.columns)` permite contar columnas.
8. Las herramientas de inspección funcionan sobre cualquier DataFrame.
9. Inspeccionar primero ayuda a detectar problemas de estructura y calidad.
10. Ninguna de estas herramientas transforma el DataFrame.